# LDCT Improved Mod-Seg-SE(2) — Multi-Task Architecture

This notebook improves the original `LDCT-mod-se-base` with:
- **Deeper encoder**: 3 levels (32→64→128 filters) + bottleneck (256 filters)
- **Batch Normalization** after every Conv block for training stability
- **Auto-mask generation** using Otsu thresholding (lung window HU windowing)
- **Data augmentation**: flip, rotation, brightness, Gaussian noise
- **Better training**: ReduceLROnPlateau + EarlyStopping + ModelCheckpoint
- **Richer evaluation**: Accuracy, ROC-AUC, Dice Score, segmentation visualization

> Base model: **Mod-Seg-SE(2)** — preserved and extended, NOT replaced.

## Step 0: Setup & Kaggle Dataset Download

In [ ]:
import os

# Setup Kaggle API Key
os.environ["KAGGLE_USERNAME"] = "muhammadaliffandy"
os.environ["KAGGLE_KEY"] = "KGAT_6cf20e173408038efc8c307643a53392"

!pip install -q kagglehub
!pip install pydicom

import kagglehub

print("⏳ Downloading CT Low Dose Reconstruction dataset...")
download_path = kagglehub.dataset_download("andrewmvd/ct-low-dose-reconstruction")
print(f"✅ Download done: {download_path}")

!cp -r "{download_path}" /content/
print("📁 Dataset copied to /content/")

## Step 1: Imports & Configuration

In [ ]:
import os
import cv2
import glob
import random
import numpy as np
import warnings
import matplotlib.pyplot as plt
import seaborn as sns
import pydicom

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    confusion_matrix, classification_report,
    accuracy_score, roc_auc_score
)
from sklearn.utils import shuffle

import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Conv2D, MaxPooling2D, UpSampling2D,
    Flatten, Dense, Concatenate, GlobalAveragePooling2D,
    Reshape, Multiply, Dropout, BatchNormalization, Activation
)
from tensorflow.keras.callbacks import (
    ReduceLROnPlateau, EarlyStopping, ModelCheckpoint
)
from tensorflow.keras.utils import to_categorical

# ─── Suppress warnings ───────────────────────────────────────────────────────
warnings.filterwarnings("ignore")
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

# ─── GPU Setup ───────────────────────────────────────────────────────────────
print("🔍 Checking GPU...")
physical_devices = tf.config.list_physical_devices('GPU')
if len(physical_devices) > 0:
    tf.config.experimental.set_memory_growth(physical_devices[0], True)
    print(f"✅ GPU Detected: {physical_devices[0].name}")
else:
    print("⚠️ No GPU detected. Training will be slow on CPU.")

# ─── Configuration ────────────────────────────────────────────────────────────
IMG_SIZE     = (256, 256)          # Input image spatial size
NUM_CLASSES  = 2                   # Full Dose (0) vs Quarter Dose (1)
CLASS_NAMES  = ['Full_Dose', 'Quarter_Dose']
BATCH_SIZE   = 16
EPOCHS       = 30
LIMIT        = 600                 # Max images per class
SAVE_DIR     = "./improved_mod_seg_se2_model"

# HU Windowing for lung tissue (standard lung window)
HU_WINDOW_CENTER = -600
HU_WINDOW_WIDTH  = 1500

os.makedirs(SAVE_DIR, exist_ok=True)
PATHS = {
    "model":          os.path.join(SAVE_DIR, "mod_seg_se2_v2.keras"),
    "best_model":     os.path.join(SAVE_DIR, "best_mod_seg_se2_v2.keras"),
    "confusion_mat":  os.path.join(SAVE_DIR, "confusion_matrix.png"),
    "seg_samples":    os.path.join(SAVE_DIR, "segmentation_samples.png"),
}

print(f"\n📁 Save directory: {SAVE_DIR}")
print(f"🏷️  Classes: {CLASS_NAMES}")
print(f"📐 Image size: {IMG_SIZE}")
print(f"📦 Batch size: {BATCH_SIZE}, Epochs: {EPOCHS}")

## Step 2: CT Preprocessing (HU Windowing + Auto-Mask)

In [ ]:
# ==============================================================================
# 2A. HU WINDOWING — Enhances lung tissue visibility
# ==============================================================================
def apply_hu_window(pixel_array, center=HU_WINDOW_CENTER, width=HU_WINDOW_WIDTH):
    """
    Apply Hounsfield Unit windowing to a CT pixel array.
    Lung window: center=-600, width=1500 → captures [-1350, 150] HU range.
    Returns a float32 array normalized to [0, 1].
    """
    img = pixel_array.astype(np.float32)
    lower = center - width / 2.0
    upper = center + width / 2.0
    img = np.clip(img, lower, upper)
    img = (img - lower) / (upper - lower)   # Normalize to [0, 1]
    return img


# ==============================================================================
# 2B. AUTO-MASK GENERATION — Otsu thresholding on lung window image
# ==============================================================================
def generate_lung_mask(img_float32):
    """
    Generate a binary lung segmentation mask using Otsu thresholding.
    This serves as a proxy mask when true annotations are unavailable.

    img_float32: grayscale image in [0, 1]
    Returns: binary mask of shape (H, W), float32 values 0.0 or 1.0
    """
    gray_uint8 = (img_float32 * 255).astype(np.uint8)

    # Apply Gaussian blur to reduce noise before thresholding
    blurred = cv2.GaussianBlur(gray_uint8, (5, 5), 0)

    # Otsu's thresholding
    _, binary = cv2.threshold(blurred, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    # Morphological operations to clean up the mask
    kernel = np.ones((5, 5), np.uint8)
    mask = cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)

    return (mask / 255.0).astype(np.float32)


# ==============================================================================
# 2C. DICOM READER with HU windowing + auto-mask
# ==============================================================================
def process_dicom(file_path, img_size=IMG_SIZE):
    """
    Read a DICOM file and return:
      - img_rgb  : (H, W, 3) float32 in [0, 1]  (HU-windowed, RGB)
      - mask     : (H, W, 1) float32 binary lung mask
    Returns (None, None) if the file cannot be read.
    """
    try:
        dcm = pydicom.dcmread(file_path)
        pixel_array = dcm.pixel_array

        # Try to apply RescaleSlope / RescaleIntercept to get real HU values
        slope     = float(getattr(dcm, 'RescaleSlope', 1))
        intercept = float(getattr(dcm, 'RescaleIntercept', 0))
        hu_array  = pixel_array * slope + intercept

        # Apply lung HU window → [0, 1]
        img_windowed = apply_hu_window(hu_array)

        # Resize to target size
        img_resized = cv2.resize(img_windowed, img_size, interpolation=cv2.INTER_LINEAR)

        # Generate auto-mask from the windowed image
        mask = generate_lung_mask(img_resized)
        mask = np.expand_dims(cv2.resize(mask, img_size), axis=-1)

        # Convert grayscale to 3-channel RGB for the CNN
        img_rgb = np.stack([img_resized] * 3, axis=-1).astype(np.float32)

        return img_rgb, mask

    except Exception as e:
        return None, None


print("✅ Preprocessing functions defined.")
print("   • apply_hu_window  — lung HU windowing (center=-600, width=1500)")
print("   • generate_lung_mask — Otsu auto-mask for segmentation training")
print("   • process_dicom     — full DICOM → (image, mask) pipeline")

## Step 3: Data Loading

In [ ]:
# ==============================================================================
# 3. DATA LOADING
# ==============================================================================
def load_dataset(base_dir, limit=LIMIT):
    """
    Scan base_dir for 'Full Dose' and 'Quarter Dose' sub-directories,
    load up to `limit` DICOM images from each, apply HU windowing,
    and generate auto-masks.

    Returns:
        images : np.ndarray (N, H, W, 3)  float32
        masks  : np.ndarray (N, H, W, 1)  float32
        labels : np.ndarray (N,)          int (0=Full, 1=Quarter)
    """
    print(f"--- Loading dataset from: {base_dir} ---")

    fd_dir = os.path.join(base_dir, "Full Dose")
    qd_dir = os.path.join(base_dir, "Quarter Dose")

    if not os.path.exists(fd_dir) or not os.path.exists(qd_dir):
        print(f"❌ Error: Missing 'Full Dose' or 'Quarter Dose' inside {base_dir}")
        return None, None, None

    exts = ("*.IMA", "*.ima", "*.dcm")
    fd_files, qd_files = [], []

    for ext in exts:
        fd_files.extend(glob.glob(os.path.join(fd_dir, "**", ext), recursive=True))
        qd_files.extend(glob.glob(os.path.join(qd_dir, "**", ext), recursive=True))

    fd_files = sorted(fd_files)[:limit]
    qd_files = sorted(qd_files)[:limit]
    print(f"📦 Found {len(fd_files)} Full Dose | {len(qd_files)} Quarter Dose files")

    images, masks, labels = [], [], []

    def _load(files, label):
        class_name = CLASS_NAMES[label]
        print(f"🚀 Processing {class_name} ({len(files)} files)...")
        for fp in files:
            img, mask = process_dicom(fp)
            if img is not None:
                images.append(img)
                masks.append(mask)
                labels.append(label)

    _load(fd_files, 0)
    _load(qd_files, 1)

    print(f"✅ Total loaded: {len(images)} images")
    return np.array(images, dtype=np.float32), np.array(masks, dtype=np.float32), np.array(labels, dtype=np.int32)


print("✅ load_dataset() defined.")

## Step 4: Data Augmentation

In [ ]:
# ==============================================================================
# 4. DATA AUGMENTATION
# Applied identically to both the image AND its segmentation mask.
# ==============================================================================
def augment(img, mask, seed=None):
    """
    Apply random augmentation to image and mask pair.

    Transforms (applied consistently to both):
      - Random horizontal flip
      - Random vertical flip
      - Random rotation ±10°
      - Random brightness/contrast jitter (image only)
      - Light Gaussian noise (image only)
    """
    rng = np.random.RandomState(seed)

    # Horizontal flip
    if rng.rand() > 0.5:
        img  = np.fliplr(img)
        mask = np.fliplr(mask)

    # Vertical flip
    if rng.rand() > 0.5:
        img  = np.flipud(img)
        mask = np.flipud(mask)

    # Random rotation ±10°
    angle = rng.uniform(-10, 10)
    h, w  = img.shape[:2]
    M = cv2.getRotationMatrix2D((w / 2, h / 2), angle, 1.0)
    img  = cv2.warpAffine(img,  M, (w, h), flags=cv2.INTER_LINEAR,
                           borderMode=cv2.BORDER_REFLECT)
    # Mask uses NEAREST interp to keep binary values
    mask_2d = cv2.warpAffine(mask[:, :, 0], M, (w, h),
                              flags=cv2.INTER_NEAREST,
                              borderMode=cv2.BORDER_REFLECT)
    mask = np.expand_dims(mask_2d, axis=-1)

    # Brightness / Contrast jitter (image only)
    if rng.rand() > 0.5:
        alpha = rng.uniform(0.85, 1.15)   # contrast
        beta  = rng.uniform(-0.1,  0.1)   # brightness
        img   = np.clip(img * alpha + beta, 0.0, 1.0)

    # Gaussian noise (image only)
    if rng.rand() > 0.5:
        noise = rng.normal(0, 0.01, img.shape).astype(np.float32)
        img   = np.clip(img + noise, 0.0, 1.0)

    return img.astype(np.float32), mask.astype(np.float32)


# ─── Keras Sequence generator ────────────────────────────────────────────────
from tensorflow.keras.utils import Sequence

class LDCTDataGenerator(Sequence):
    """
    Memory-efficient generator that loads pre-loaded numpy arrays in batches,
    with optional on-the-fly augmentation.
    """
    def __init__(self, X_img, Y_mask, Y_cls, batch_size=BATCH_SIZE, augment_data=False):
        self.X_img       = X_img
        self.Y_mask      = Y_mask
        self.Y_cls       = Y_cls
        self.batch_size  = batch_size
        self.augment     = augment_data
        self.indices     = np.arange(len(X_img))

    def __len__(self):
        return int(np.ceil(len(self.X_img) / self.batch_size))

    def on_epoch_end(self):
        # Shuffle at end of each epoch
        np.random.shuffle(self.indices)

    def __getitem__(self, idx):
        batch_idx = self.indices[idx * self.batch_size : (idx + 1) * self.batch_size]
        imgs  = self.X_img[batch_idx].copy()
        masks = self.Y_mask[batch_idx].copy()
        cls   = self.Y_cls[batch_idx]

        if self.augment:
            for i in range(len(imgs)):
                imgs[i], masks[i] = augment(imgs[i], masks[i])

        return imgs, {"class_output": cls, "seg_output": masks}


print("✅ Augmentation + LDCTDataGenerator defined.")

## Step 5: Model Architecture — Improved Mod-Seg-SE(2) v2

In [ ]:
# ==============================================================================
# 5A. SQUEEZE-AND-EXCITATION BLOCK  (unchanged from original Mod-Seg-SE base)
# ==============================================================================
def se_block(input_tensor, ratio=8):
    """
    Squeeze-and-Excitation block.
    Recalibrates channel-wise feature responses by:
      1. Squeeze: Global Average Pooling → channel descriptor
      2. Excitation: Two Dense layers (bottleneck) → channel weights
      3. Scale: Multiply original feature map by channel weights
    """
    filters = input_tensor.shape[-1]

    # Squeeze
    se = GlobalAveragePooling2D()(input_tensor)
    se = Reshape((1, 1, filters))(se)

    # Excitation
    se = Dense(filters // ratio, activation='relu',
               kernel_initializer='he_normal', use_bias=False)(se)
    se = Dense(filters, activation='sigmoid',
               kernel_initializer='he_normal', use_bias=False)(se)

    # Scale
    return Multiply()([input_tensor, se])


# ==============================================================================
# 5B. CONV BLOCK HELPER — Conv → BN → ReLU  (NEW: adds BatchNorm)
# ==============================================================================
def conv_bn_relu(x, filters, kernel_size=(3, 3)):
    """Conv2D → BatchNormalization → ReLU activation."""
    x = Conv2D(filters, kernel_size, padding='same',
               kernel_initializer='he_normal', use_bias=False)(x)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    return x


# ==============================================================================
# 5C. IMPROVED Mod-Seg-SE(2) v2 — 3-level encoder + deeper decoder
# ==============================================================================
def build_mod_seg_se2_v2(input_shape=(256, 256, 3), num_classes=NUM_CLASSES):
    """
    Improved Multi-Task Mod-Seg-SE(2) Architecture.

    Improvements over original:
      - 3 encoder levels instead of 2 (adds 128-filter level)
      - Bottleneck at 256 filters
      - BatchNormalization after every Conv block
      - SE blocks at all encoder + decoder levels
      - Deeper classification head (128 → 64 → num_classes)
      - 3 decoder levels matching new encoder depth
    """
    print("--- Building Improved Mod-Seg-SE(2) v2 ---")

    inputs = Input(shape=input_shape, name="image_input")

    # ── ENCODER Level 1: 32 filters ──────────────────────────────────────────
    c1 = conv_bn_relu(inputs, 32)
    c1 = conv_bn_relu(c1, 32)
    c1 = se_block(c1)                        # SE attention
    p1 = MaxPooling2D((2, 2))(c1)            # 128×128

    # ── ENCODER Level 2: 64 filters ──────────────────────────────────────────
    c2 = conv_bn_relu(p1, 64)
    c2 = conv_bn_relu(c2, 64)
    c2 = se_block(c2)
    p2 = MaxPooling2D((2, 2))(c2)            # 64×64

    # ── ENCODER Level 3: 128 filters (NEW) ────────────────────────────────────
    c3 = conv_bn_relu(p2, 128)
    c3 = conv_bn_relu(c3, 128)
    c3 = se_block(c3)
    p3 = MaxPooling2D((2, 2))(c3)            # 32×32

    # ── BOTTLENECK: 256 filters ───────────────────────────────────────────────
    bn = conv_bn_relu(p3, 256)
    bn = conv_bn_relu(bn, 256)
    bn = se_block(bn)                        # 32×32

    # ── BRANCH 1: CLASSIFICATION HEAD ────────────────────────────────────────
    cls = GlobalAveragePooling2D(name="gap_classification")(bn)
    cls = Dense(128, activation='relu', kernel_initializer='he_normal')(cls)
    cls = BatchNormalization()(cls)
    cls = Dropout(0.4)(cls)
    cls = Dense(64, activation='relu', kernel_initializer='he_normal')(cls)
    cls = Dropout(0.3)(cls)
    class_output = Dense(
        num_classes, activation='softmax', name="class_output"
    )(cls)

    # ── BRANCH 2: DECODER (Segmentation) ─────────────────────────────────────
    # Decoder Level 3 → 128 filters
    u3 = UpSampling2D((2, 2))(bn)            # 64×64
    d3 = Concatenate()([u3, c3])
    d3 = conv_bn_relu(d3, 128)
    d3 = conv_bn_relu(d3, 128)
    d3 = se_block(d3)

    # Decoder Level 2 → 64 filters
    u2 = UpSampling2D((2, 2))(d3)            # 128×128
    d2 = Concatenate()([u2, c2])
    d2 = conv_bn_relu(d2, 64)
    d2 = conv_bn_relu(d2, 64)
    d2 = se_block(d2)

    # Decoder Level 1 → 32 filters
    u1 = UpSampling2D((2, 2))(d2)            # 256×256
    d1 = Concatenate()([u1, c1])
    d1 = conv_bn_relu(d1, 32)
    d1 = conv_bn_relu(d1, 32)
    d1 = se_block(d1)

    # Binary segmentation output (1 channel, sigmoid)
    seg_output = Conv2D(
        1, (1, 1), activation='sigmoid', name="seg_output"
    )(d1)

    # ── COMPILE ───────────────────────────────────────────────────────────────
    model = Model(inputs=inputs, outputs=[class_output, seg_output])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss={
            "class_output": "categorical_crossentropy",
            "seg_output":   "binary_crossentropy"
        },
        loss_weights={
            "class_output": 1.0,
            "seg_output":   0.5    # Slightly lower weight — classification is primary
        },
        metrics={
            "class_output": ["accuracy"],
            "seg_output":   ["accuracy"]
        }
    )

    return model


# ─── Quick architecture sanity check ─────────────────────────────────────────
model = build_mod_seg_se2_v2(input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3))
model.summary()
print(f"\n✅ Model built. Total params: {model.count_params():,}")

## Step 6: Load Data & Quick Verification

In [ ]:
# ─── Point to the dataset directory ──────────────────────────────────────────
BASE_DIR = "/content/CT_low_dose_reconstruction_dataset/Original Data"

X_img, Y_mask, y_labels = load_dataset(BASE_DIR, limit=LIMIT)

if X_img is None:
    raise RuntimeError(f"❌ Dataset not found at {BASE_DIR}. Adjust BASE_DIR.")

print(f"\n📊 Dataset shapes:")
print(f"   Images : {X_img.shape}  dtype={X_img.dtype}")
print(f"   Masks  : {Y_mask.shape} dtype={Y_mask.dtype}")
print(f"   Labels : {y_labels.shape} — {dict(zip(*np.unique(y_labels, return_counts=True)))}")

# ─── Visualize a few preprocessing results ───────────────────────────────────
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
fig.suptitle("Preprocessing Verification (HU Windowed Image + Auto-Mask)", fontsize=13)
sample_idx = np.random.choice(len(X_img), 4, replace=False)

for col, idx in enumerate(sample_idx):
    axes[0, col].imshow(X_img[idx], cmap='gray')
    axes[0, col].set_title(f"Image [{CLASS_NAMES[y_labels[idx]]}]")
    axes[0, col].axis('off')

    axes[1, col].imshow(Y_mask[idx, :, :, 0], cmap='hot')
    axes[1, col].set_title("Auto-Mask (Otsu)")
    axes[1, col].axis('off')

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "preprocessing_preview.png"), dpi=100)
plt.show()
print("✅ Preprocessing preview saved.")

## Step 7: Train/Val Split & Training Pipeline

In [ ]:
# ─── Shuffle & Split ─────────────────────────────────────────────────────────
print("🔀 Shuffling dataset...")
X_img, Y_mask, y_labels = shuffle(X_img, Y_mask, y_labels, random_state=42)
y_cat = to_categorical(y_labels, num_classes=NUM_CLASSES)

(
    X_train, X_test,
    Ym_train, Ym_test,
    Yc_train, Yc_test,
    yl_train, yl_test
) = train_test_split(
    X_img, Y_mask, y_cat, y_labels,
    test_size=0.2, random_state=42, stratify=y_labels
)

print(f"Training samples : {len(X_train)}")
print(f"Testing samples  : {len(X_test)}")

# ─── Generators ──────────────────────────────────────────────────────────────
train_gen = LDCTDataGenerator(X_train, Ym_train, Yc_train,
                               batch_size=BATCH_SIZE, augment_data=True)
val_gen   = LDCTDataGenerator(X_test,  Ym_test,  Yc_test,
                               batch_size=BATCH_SIZE, augment_data=False)

# ─── Callbacks ───────────────────────────────────────────────────────────────
callbacks = [
    # Save the best model by validation classification accuracy
    ModelCheckpoint(
        PATHS["best_model"],
        monitor='val_class_output_accuracy',
        save_best_only=True,
        mode='max',
        verbose=1
    ),
    # Reduce LR if val loss plateaus for 5 epochs
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=5,
        min_lr=1e-6,
        verbose=1
    ),
    # Stop training early if val_class_accuracy does not improve for 10 epochs
    EarlyStopping(
        monitor='val_class_output_accuracy',
        patience=10,
        restore_best_weights=True,
        mode='max',
        verbose=1
    ),
]

# ─── Train ───────────────────────────────────────────────────────────────────
print("\n--- Training Multi-Task Mod-Seg-SE(2) v2 ---")
with tf.device('/GPU:0'):
    history = model.fit(
        train_gen,
        validation_data=val_gen,
        epochs=EPOCHS,
        callbacks=callbacks,
        verbose=1
    )

# ─── Save final model ────────────────────────────────────────────────────────
model.save(PATHS["model"])
print(f"\n✅ Final model saved → {PATHS['model']}")
print(f"✅ Best model saved  → {PATHS['best_model']}")

## Step 8: Training Curves

In [ ]:
# ─── Plot training history ────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("Training History — Mod-Seg-SE(2) v2", fontsize=14)

epochs_range = range(1, len(history.history['class_output_accuracy']) + 1)

# Classification accuracy
axes[0, 0].plot(epochs_range, history.history['class_output_accuracy'], label='Train')
axes[0, 0].plot(epochs_range, history.history['val_class_output_accuracy'], label='Val')
axes[0, 0].set_title('Classification Accuracy')
axes[0, 0].set_xlabel('Epoch'); axes[0, 0].set_ylabel('Accuracy')
axes[0, 0].legend(); axes[0, 0].grid(True)

# Classification loss
axes[0, 1].plot(epochs_range, history.history['class_output_loss'], label='Train')
axes[0, 1].plot(epochs_range, history.history['val_class_output_loss'], label='Val')
axes[0, 1].set_title('Classification Loss')
axes[0, 1].set_xlabel('Epoch'); axes[0, 1].set_ylabel('Loss')
axes[0, 1].legend(); axes[0, 1].grid(True)

# Segmentation accuracy
axes[1, 0].plot(epochs_range, history.history['seg_output_accuracy'], label='Train')
axes[1, 0].plot(epochs_range, history.history['val_seg_output_accuracy'], label='Val')
axes[1, 0].set_title('Segmentation Accuracy')
axes[1, 0].set_xlabel('Epoch'); axes[1, 0].set_ylabel('Accuracy')
axes[1, 0].legend(); axes[1, 0].grid(True)

# Total loss
axes[1, 1].plot(epochs_range, history.history['loss'], label='Train')
axes[1, 1].plot(epochs_range, history.history['val_loss'], label='Val')
axes[1, 1].set_title('Total Loss')
axes[1, 1].set_xlabel('Epoch'); axes[1, 1].set_ylabel('Loss')
axes[1, 1].legend(); axes[1, 1].grid(True)

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'training_curves.png'), dpi=100)
plt.show()
print("✅ Training curves saved.")

## Step 9: Evaluation (Accuracy + AUC + Dice + Visualizations)

In [ ]:
# ─── Inference on test set ────────────────────────────────────────────────────
print("--- Evaluating on Test Set ---")
preds = model.predict(X_test, batch_size=BATCH_SIZE, verbose=1)
cls_proba  = preds[0]    # (N, NUM_CLASSES)
seg_preds  = preds[1]    # (N, H, W, 1)

y_pred_labels = np.argmax(cls_proba, axis=1)
y_true_labels = np.argmax(Yc_test,   axis=1)

# ─── Classification Metrics ───────────────────────────────────────────────────
acc = accuracy_score(y_true_labels, y_pred_labels) * 100

# ROC-AUC (binary)
auc = roc_auc_score(y_true_labels, cls_proba[:, 1])

print("\n" + "="*55)
print(f"  🎯 CLASSIFICATION TEST ACCURACY : {acc:.2f}%")
print(f"  📈 ROC-AUC                      : {auc:.4f}")
print("="*55 + "\n")
print("Classification Report:")
print(classification_report(y_true_labels, y_pred_labels, target_names=CLASS_NAMES))


# ─── Segmentation Dice Score ──────────────────────────────────────────────────
def dice_score(y_true, y_pred, threshold=0.5, smooth=1e-6):
    """Pixel-wise Dice coefficient."""
    y_pred_bin = (y_pred > threshold).astype(np.float32)
    intersection = np.sum(y_true * y_pred_bin)
    union = np.sum(y_true) + np.sum(y_pred_bin)
    return (2.0 * intersection + smooth) / (union + smooth)

dice = dice_score(Ym_test, seg_preds)
print(f"  🧩 Segmentation Dice Score      : {dice:.4f}")


# ─── Confusion Matrix ─────────────────────────────────────────────────────────
cm = confusion_matrix(y_true_labels, y_pred_labels)
plt.figure(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.title(f'Confusion Matrix — Acc: {acc:.2f}%  |  AUC: {auc:.4f}')
plt.ylabel('True Label'); plt.xlabel('Predicted Label')
plt.tight_layout()
plt.savefig(PATHS["confusion_mat"], dpi=100)
plt.show()
print(f"✅ Confusion matrix saved → {PATHS['confusion_mat']}")


# ─── Segmentation Overlay Visualization ──────────────────────────────────────
n_samples = min(5, len(X_test))
sample_idx = np.random.choice(len(X_test), n_samples, replace=False)

fig, axes = plt.subplots(n_samples, 3, figsize=(12, 4 * n_samples))
fig.suptitle("Segmentation Visualization — CT | Auto-Mask | Predicted Mask", fontsize=13)

for row, idx in enumerate(sample_idx):
    pred_label = CLASS_NAMES[y_pred_labels[idx]]
    true_label = CLASS_NAMES[y_true_labels[idx]]
    correct    = "✅" if y_pred_labels[idx] == y_true_labels[idx] else "❌"

    axes[row, 0].imshow(X_test[idx], cmap='gray')
    axes[row, 0].set_title(f"CT Image\nTrue: {true_label} | Pred: {pred_label} {correct}")
    axes[row, 0].axis('off')

    axes[row, 1].imshow(Ym_test[idx, :, :, 0], cmap='hot')
    axes[row, 1].set_title("Auto-Mask (Ground Truth)")
    axes[row, 1].axis('off')

    axes[row, 2].imshow(seg_preds[idx, :, :, 0], cmap='hot')
    axes[row, 2].set_title(f"Predicted Mask (Dice={dice:.3f})")
    axes[row, 2].axis('off')

plt.tight_layout()
plt.savefig(PATHS["seg_samples"], dpi=100)
plt.show()
print(f"✅ Segmentation samples saved → {PATHS['seg_samples']}")
print("\n🎉 IMPROVED MULTI-TASK PIPELINE COMPLETE!")

## Step 9b: GradCAM — Where Does the Model Look to Decide?

Grad-CAM (Gradient-weighted Class Activation Mapping) visualizes **which regions of the CT slice the model attends to** when making its classification decision (Full Dose vs Quarter Dose).

This is the true answer to *"where is the area that drives the decision?"* — complementing the segmentation mask.

In [ ]:
# ==============================================================================
# STEP 9b: GradCAM VISUALIZATION
# Highlights the DECISION REGIONS used by the classification head.
# ==============================================================================
import tensorflow as tf
import numpy as np
import cv2
import matplotlib.pyplot as plt
import matplotlib.cm as cm


# ─── GradCAM Engine ────────────────────────────────────────────────────────────
def make_gradcam_heatmap(img_array, model, last_conv_layer_name, pred_index=None):
    """
    Compute GradCAM heatmap for a single image.

    Args:
        img_array          : (1, H, W, 3) float32 in [0, 1]
        model              : Keras multi-task Model
        last_conv_layer_name: Name of the last Conv2D layer BEFORE the bottleneck
        pred_index         : Class index to explain (None = argmax of prediction)

    Returns:
        heatmap : (H, W) float32 in [0, 1]
        pred_class : int — predicted class index
        pred_prob  : float — probability of predicted class
    """
    # Build a sub-model that outputs:
    #   - the activations of the target conv layer
    #   - the final classification output
    grad_model = tf.keras.models.Model(
        inputs=model.inputs,
        outputs=[
            model.get_layer(last_conv_layer_name).output,
            model.get_layer('class_output').output
        ]
    )

    with tf.GradientTape() as tape:
        tape.watch(img_array)
        conv_outputs, predictions = grad_model(img_array, training=False)

        if pred_index is None:
            pred_index = tf.argmax(predictions[0])
        pred_prob = float(predictions[0][pred_index])

        # Score for the target class
        class_channel = predictions[:, pred_index]

    # Gradients of class score w.r.t. conv feature map
    grads = tape.gradient(class_channel, conv_outputs)   # (1, h, w, C)

    # Global Average Pooling of gradients → importance weights per channel
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))  # (C,)

    # Weight channels by their importance
    conv_outputs = conv_outputs[0]                         # (h, w, C)
    heatmap = conv_outputs @ pooled_grads[..., tf.newaxis] # (h, w, 1)
    heatmap = tf.squeeze(heatmap)                          # (h, w)

    # ReLU + normalize to [0, 1]
    heatmap = tf.maximum(heatmap, 0) / (tf.math.reduce_max(heatmap) + 1e-8)
    heatmap = heatmap.numpy()

    return heatmap, int(pred_index), pred_prob


def overlay_gradcam(img_float32, heatmap, alpha=0.45, colormap=cv2.COLORMAP_JET):
    """
    Overlay a GradCAM heatmap on the original image.

    Returns: (H, W, 3) uint8 blended image.
    """
    # Upscale heatmap to image size
    heatmap_up = cv2.resize(heatmap, (img_float32.shape[1], img_float32.shape[0]))

    # Convert heatmap to BGR colourmap (uint8)
    heatmap_color = cv2.applyColorMap(np.uint8(255 * heatmap_up), colormap)
    heatmap_color = cv2.cvtColor(heatmap_color, cv2.COLOR_BGR2RGB)

    # Convert original image to uint8 RGB
    img_uint8 = np.uint8(255 * img_float32)
    if img_uint8.shape[-1] == 1:
        img_uint8 = np.concatenate([img_uint8] * 3, axis=-1)

    # Blend
    superimposed = cv2.addWeighted(heatmap_color, alpha, img_uint8, 1 - alpha, 0)
    return superimposed


# ─── Find the last Conv2D layer name (before bottleneck SE block) ─────────────
# The bottleneck is the last encoder stage — we want the last conv output at
# the deepest spatial resolution (32x32 for 256-input with 3 MaxPools).
# We scan for the last conv layer whose output spatial size matches 32x32.
target_conv_layer = None
for layer in model.layers:
    if isinstance(layer, tf.keras.layers.Conv2D):
        try:
            out_shape = layer.output.shape
        except Exception:
            continue
        if out_shape[1] == IMG_SIZE[0] // 8:  # 256 // 8 = 32
            target_conv_layer = layer.name

# ── Fallback 1: find the last Conv2D layer with the most filters (= bottleneck)
if target_conv_layer is None:
    print("⚠️  Shape scan failed — falling back to highest-filter Conv2D")
    max_filters = 0
    for layer in model.layers:
        if isinstance(layer, tf.keras.layers.Conv2D):
            f = layer.filters
            if f >= max_filters:
                max_filters = f
                target_conv_layer = layer.name

# ── Fallback 2: just use the last Conv2D in the model
if target_conv_layer is None:
    print("⚠️  Filter fallback failed — using last Conv2D in model")
    for layer in reversed(model.layers):
        if isinstance(layer, tf.keras.layers.Conv2D):
            target_conv_layer = layer.name
            break

print(f"🎯 Target Conv layer for GradCAM: '{target_conv_layer}'")
if target_conv_layer is None:
    raise RuntimeError("No Conv2D layers in model. Cannot compute GradCAM.")


# ─── Visualize GradCAM + Segmentation side-by-side ────────────────────────────
n_vis = min(6, len(X_test))
sample_idx = np.random.choice(len(X_test), n_vis, replace=False)

fig, axes = plt.subplots(n_vis, 4, figsize=(18, 5 * n_vis))
fig.suptitle(
    "GradCAM Decision Map vs Segmentation Output\n"
    "Col 1: CT (HU-windowed)  |  Col 2: GradCAM (what model attends to)  "
    "|  Col 3: Predicted Segmentation  |  Col 4: Overlay (GradCAM + Seg)",
    fontsize=12
)

for row, idx in enumerate(sample_idx):
    img      = X_test[idx]                    # (H, W, 3) float32
    true_lbl = CLASS_NAMES[y_true_labels[idx]]
    pred_lbl = CLASS_NAMES[y_pred_labels[idx]]
    correct  = "✅" if y_true_labels[idx] == y_pred_labels[idx] else "❌"

    # ── GradCAM ────────────────────────────────────────────────────────────────
    img_batch = tf.cast(img[np.newaxis, ...], tf.float32)
    heatmap, pred_cls, pred_prob = make_gradcam_heatmap(
        img_batch, model, target_conv_layer, pred_index=y_pred_labels[idx]
    )
    gradcam_overlay = overlay_gradcam(img, heatmap)  # (H, W, 3) uint8

    # ── Predicted segmentation mask ────────────────────────────────────────────
    seg_mask = seg_preds[idx, :, :, 0]               # (H, W) float32
    seg_binary = (seg_mask > 0.5).astype(np.float32)

    # ── GradCAM masked to segmentation area (focus overlay) ───────────────────
    masked_heatmap = heatmap * cv2.resize(seg_binary, heatmap.shape[::-1])
    # Upscale and overlay
    seg_focus_overlay = overlay_gradcam(img, masked_heatmap, alpha=0.55)

    # ── Plot ───────────────────────────────────────────────────────────────────
    # Col 0: Original CT
    axes[row, 0].imshow(img, cmap='gray')
    axes[row, 0].set_title(f"CT Image\nTrue: {true_lbl}\nPred: {pred_lbl} {correct}\n(p={pred_prob:.3f})", fontsize=9)
    axes[row, 0].axis('off')

    # Col 1: GradCAM heatmap overlay — full attention map
    axes[row, 1].imshow(gradcam_overlay)
    axes[row, 1].set_title("GradCAM\n(Decision attention map)\nHot=high attention", fontsize=9)
    axes[row, 1].axis('off')

    # Col 2: Predicted segmentation mask
    axes[row, 2].imshow(img, cmap='gray')
    axes[row, 2].imshow(seg_mask, cmap='hot', alpha=0.5)
    axes[row, 2].set_title(f"Segmentation Mask\n(Dice={dice:.3f})", fontsize=9)
    axes[row, 2].axis('off')

    # Col 3: GradCAM restricted to segmented region = focused decision area
    axes[row, 3].imshow(seg_focus_overlay)
    axes[row, 3].set_title("Segmentation × GradCAM\n(Decision within seg. region)", fontsize=9)
    axes[row, 3].axis('off')


plt.tight_layout()
gradcam_path = os.path.join(SAVE_DIR, 'gradcam_analysis.png')
plt.savefig(gradcam_path, dpi=100, bbox_inches='tight')
plt.show()
print(f"✅ GradCAM analysis saved → {gradcam_path}")


# ─── Summary: Top attending region per class ─────────────────────────────────
print("\n📊 GradCAM Region Analysis per Class:")
print("-" * 55)

for cls_idx, cls_name in enumerate(CLASS_NAMES):
    class_mask = y_true_labels == cls_idx
    if not class_mask.any():
        continue

    # Compute mean heatmap across images of this class
    heatmaps = []
    for i in np.where(class_mask)[0][:20]:   # Use up to 20 samples
        img_b = tf.cast(X_test[i][np.newaxis], tf.float32)
        hm, _, _ = make_gradcam_heatmap(img_b, model, target_conv_layer, pred_index=cls_idx)
        heatmaps.append(cv2.resize(hm, IMG_SIZE))

    mean_hm = np.mean(heatmaps, axis=0)   # (H, W)

    # Find the peak attention centroid
    peak_y, peak_x = np.unravel_index(np.argmax(mean_hm), mean_hm.shape)
    pct_y = peak_y / IMG_SIZE[0] * 100
    pct_x = peak_x / IMG_SIZE[1] * 100

    # Fraction of attention in top 10% of heatmap values
    threshold = np.percentile(mean_hm, 90)
    focused_area_pct = (mean_hm > threshold).sum() / mean_hm.size * 100

    print(f"  [{cls_name}]")
    print(f"    Peak attention at : ({peak_x:3d}px, {peak_y:3d}px)  "
          f"→ ({pct_x:.0f}% from left, {pct_y:.0f}% from top)")
    print(f"    High-attention area: {focused_area_pct:.1f}% of image")
    print()

print("✅ GradCAM analysis complete.")

## Step 9c: GradCAM Mean Attention Map per Class

Visualize the **average attention map** across all test images of each class — this shows *systematic* region differences between Full Dose and Quarter Dose.

In [ ]:
# ==============================================================================
# STEP 9c: MEAN GradCAM PER CLASS — What does the model focus on systematically?
# ==============================================================================
fig, axes = plt.subplots(1, len(CLASS_NAMES), figsize=(6 * len(CLASS_NAMES), 5))
if len(CLASS_NAMES) == 1:
    axes = [axes]
fig.suptitle("Mean GradCAM Attention Map per Class\n"
             "(Averaged over up to 30 test images each)", fontsize=13)

all_mean_heatmaps = {}

for cls_idx, cls_name in enumerate(CLASS_NAMES):
    class_sample_idx = np.where(y_true_labels == cls_idx)[0][:30]
    if len(class_sample_idx) == 0:
        continue

    heatmaps = []
    for i in class_sample_idx:
        img_b = tf.cast(X_test[i][np.newaxis], tf.float32)
        hm, _, _ = make_gradcam_heatmap(img_b, model, target_conv_layer, pred_index=cls_idx)
        heatmaps.append(cv2.resize(hm, IMG_SIZE))

    mean_hm = np.mean(heatmaps, axis=0)   # (H, W)
    mean_hm = (mean_hm - mean_hm.min()) / (mean_hm.max() - mean_hm.min() + 1e-8)
    all_mean_heatmaps[cls_name] = mean_hm

    # Overlay on mean CT image of that class
    mean_ct = np.mean(X_test[class_sample_idx], axis=0)   # (H, W, 3)
    overlay  = overlay_gradcam(mean_ct, mean_hm, alpha=0.5)

    ax = axes[cls_idx]
    ax.imshow(overlay)
    ax.set_title(f"{cls_name}\n({len(class_sample_idx)} samples averaged)", fontsize=11)
    ax.axis('off')

plt.tight_layout()
mean_gradcam_path = os.path.join(SAVE_DIR, 'mean_gradcam_per_class.png')
plt.savefig(mean_gradcam_path, dpi=100, bbox_inches='tight')
plt.show()
print(f"✅ Mean GradCAM per class saved → {mean_gradcam_path}")


# ─── Difference map: Full Dose attention vs Quarter Dose attention ─────────────
if len(all_mean_heatmaps) == 2:
    hm_fd = all_mean_heatmaps.get('Full_Dose',    np.zeros(IMG_SIZE))
    hm_qd = all_mean_heatmaps.get('Quarter_Dose', np.zeros(IMG_SIZE))
    diff_map = hm_fd - hm_qd   # Positive = FD-unique attention; Negative = QD-unique

    fig, ax = plt.subplots(1, 1, figsize=(7, 6))
    im = ax.imshow(diff_map, cmap='RdBu_r', vmin=-1, vmax=1)
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04,
                 label='Full_Dose attention  ←  →  Quarter_Dose attention')
    ax.set_title("Attention Difference Map\n"
                 "Blue = model attends here for Quarter_Dose\n"
                 "Red  = model attends here for Full_Dose", fontsize=11)
    ax.axis('off')
    plt.tight_layout()
    diff_path = os.path.join(SAVE_DIR, 'gradcam_attention_diff.png')
    plt.savefig(diff_path, dpi=100, bbox_inches='tight')
    plt.show()
    print(f"✅ Attention difference map saved → {diff_path}")
    print()
    print("📌 Interpretation guide:")
    print("   RED regions  → the model sees these areas as indicative of FULL DOSE")
    print("   BLUE regions → the model attends here when detecting QUARTER DOSE")
    print("   WHITE/grey   → regions used equally for both classes")


## Step 10: Save to Google Drive (Optional)

In [ ]:
# Uncomment to save model artifacts to your Google Drive
# from google.colab import drive
# drive.mount('/content/drive')
# !cp -r "{SAVE_DIR}" /content/drive/MyDrive/ldct_improved_se2
# print("✅ Model saved to Google Drive.")
print("Drive save step ready — uncomment above lines to enable.")